# Quickstart: cumulative returns from Global Markets OHLCV

Demo notebook for the [**Global Markets OHLCV Dataset**](https://www.kaggle.com/datasets/benjaminpo/finance-dataset).

It loads a few liquid daily series (US equities, S&P 500, Bitcoin, EUR/USD), computes simple returns, and plots **cumulative growth of $1**.

Pipeline / listings: [benjaminpo/finance-dataset](https://github.com/benjaminpo/finance-dataset)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.options.display.max_rows = 8
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")


def find_data_dir() -> Path:
    """Resolve the dataset root on Kaggle or a local checkout."""
    candidates = [
        Path("/kaggle/input/finance-dataset"),
        Path("/kaggle/input/benjaminpo-finance-dataset"),
    ]
    for root in Path("/kaggle/input").glob("*") if Path("/kaggle/input").is_dir() else []:
        candidates.append(root)

    # Local fallbacks when opening this notebook from the GitHub repo.
    here = Path.cwd()
    candidates.extend(
        [
            here / "data",
            here.parent / "data",
            here.parent.parent / "data",
        ]
    )

    for path in candidates:
        if (path / "stocks_us" / "1d").is_dir() or (path / "crypto" / "1d").is_dir():
            return path

    raise FileNotFoundError(
        "Could not find dataset root. Attach benjaminpo/finance-dataset on Kaggle, "
        "or run with a local data/ directory from the finance-dataset repo."
    )


DATA_DIR = find_data_dir()
print("DATA_DIR =", DATA_DIR)

In [ ]:
def safe_filename(ticker: str) -> str:
    """Match the pipeline's on-disk naming (^GSPC → GSPC.csv, EURUSD=X → EURUSD_X.csv)."""
    return ticker.replace("^", "").replace("=", "_").replace("/", "_")


def load_close(asset_class: str, ticker: str, interval: str = "1d") -> pd.Series:
    path = DATA_DIR / asset_class / interval / f"{safe_filename(ticker)}.csv"
    df = pd.read_csv(path, parse_dates=["Datetime"], index_col="Datetime")
    col = "Adj Close" if "Adj Close" in df.columns else "Close"
    s = df[col].astype(float).sort_index()
    s.name = ticker
    return s


# A small liquid basket across asset classes in this dataset.
SERIES = [
    ("stocks_us", "AAPL"),
    ("stocks_us", "MSFT"),
    ("indices", "^GSPC"),
    ("crypto", "BTC-USD"),
    ("currencies", "EURUSD=X"),
]

closes = pd.concat(
    [load_close(asset_class, ticker) for asset_class, ticker in SERIES],
    axis=1,
)

# Align on shared calendar; FX/crypto trade more days than equities.
closes = closes.dropna(how="any")
closes.tail()

In [ ]:
LOOKBACK_DAYS = 365 * 3  # last ~3 years of overlapping history

window = closes.iloc[-LOOKBACK_DAYS:]
returns = window.pct_change().dropna(how="any")
cum_growth = (1.0 + returns).cumprod()

print(
    f"Window: {window.index.min().date()} → {window.index.max().date()} "
    f"({len(window)} bars)"
)
returns.describe().T[["mean", "std", "min", "max"]]

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True, constrained_layout=True)

window.div(window.iloc[0]).plot(ax=axes[0], lw=1.4)
axes[0].set_title("Normalized price (start = 1.0)")
axes[0].set_ylabel("Index")
axes[0].legend(loc="upper left", fontsize=9)

cum_growth.plot(ax=axes[1], lw=1.4)
axes[1].set_title("Growth of $1 (daily simple returns, compounded)")
axes[1].set_ylabel("Portfolio value")
axes[1].set_xlabel("Datetime (UTC)")
axes[1].legend(loc="upper left", fontsize=9)

plt.show()

In [ ]:
corr = returns.corr()

fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.index)
ax.set_title("Daily return correlation")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=8)

plt.show()
corr

## Try other tickers

Paths follow `{asset_class}/{interval}/{TICKER}.csv`:

| Example | Path |
|---|---|
| Apple daily | `stocks_us/1d/AAPL.csv` |
| Bitcoin daily | `crypto/1d/BTC-USD.csv` |
| S&P 500 | `indices/1d/GSPC.csv` (`^` stripped) |
| EUR/USD | `currencies/1d/EURUSD_X.csv` (`=` → `_`) |
| AAPL 5-minute snapshot | `stocks_us/5m/AAPL_YYYY-MM-DD.csv` |

Asset classes: `stocks_us`, `stocks_kr`, `stocks_jp`, `stocks_eu`, `stocks_hk`, `indices`, `rates`, `futures`, `crypto`, `currencies`.